# Calculate accuracy for retained MSCAF-TransUNet artifacts

This notebook performs post-processing only. It does not train a model and does not need a GPU.

It reads exported experiment zip files from Google Drive, pairs `*_pred.nii.gz` with `*_gt.nii.gz`, and writes:

- `accuracy_results.json`
- `accuracy_summary.md`

Reported metrics:

- `voxel_accuracy`: all correctly classified voxels divided by all voxels, including background
- `foreground_voxel_accuracy`: correctly classified organ voxels divided by all ground-truth organ voxels
- `mean_foreground_accuracy`: macro average of recall across the 8 Synapse organs
- `pancreas_accuracy`: recall for Synapse class `6`

`mean_foreground_accuracy` and `pancreas_accuracy` are the useful report metrics. `voxel_accuracy` is retained for completeness because background voxels can make it look deceptively high.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/ihatesea69/MSCAF-TransUNet.git'
REPO_BRANCH = 'codex/cleanup-research-artifacts'  # Change to 'main' after PR #1 is merged.
PROJECT_DIR = Path('/content/MSCAF-TransUNet')
DRIVE_EXPORT_DIR = Path('/content/drive/MyDrive/transunet_colab_outputs')
OUTPUT_DIR = DRIVE_EXPORT_DIR / 'accuracy_exports'

# Explicit mappings are required when two runs share the same snapshot filename.
# The current Drive zip with this filename is expected to be the latest completed run 02.
ARTIFACT_MAP = {
    'pre_hidden_1_16_r16_run_02': DRIVE_EXPORT_DIR / 'TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224_attn-pre_hidden-1_16-r16.zip',
    # Add a preserved run 01 zip here if it exists under a different filename:
    # 'pre_hidden_1_16_r16_run_01': DRIVE_EXPORT_DIR / 'preserved-run-01.zip',
    # Add external artifacts here if retained predictions are available:
    # 'baseline_reproduction': DRIVE_EXPORT_DIR / 'baseline-reproduction.zip',
    # 'mscaf_cnn_fusion_3scale': DRIVE_EXPORT_DIR / 'mscaf-cnn-fusion-3scale.zip',
}


In [ ]:
import shutil
import subprocess

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)],
    check=True,
)
subprocess.run(
    ['python', '-m', 'pip', 'install', '-q', 'numpy>=1.26,<2', 'SimpleITK'],
    check=True,
)


In [ ]:
import json
import subprocess
import sys

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
artifact_map_path = OUTPUT_DIR / 'artifact_map.json'
artifact_map_path.write_text(
    json.dumps({run_id: str(path) for run_id, path in ARTIFACT_MAP.items()}, indent=2),
    encoding='utf-8',
)

subprocess.run(
    [
        sys.executable,
        str(PROJECT_DIR / 'scripts' / 'calculate_accuracy_from_artifacts.py'),
        '--registry', str(PROJECT_DIR / 'docs' / 'results' / 'run_registry.json'),
        '--artifact-dir', str(DRIVE_EXPORT_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--artifact-map', str(artifact_map_path),
    ],
    check=True,
)


In [ ]:
from IPython.display import Markdown, display

summary_path = OUTPUT_DIR / 'accuracy_summary.md'
display(Markdown(summary_path.read_text(encoding='utf-8')))
print('JSON:', OUTPUT_DIR / 'accuracy_results.json')
print('Markdown:', summary_path)


## Interpreting missing rows

- `missing`: add a preserved zip path to `ARTIFACT_MAP`, or rerun only the evaluation cell of the original experiment notebook with `SAVE_NIFTI = True`.
- `shared artifact filename is ambiguous`: two runs exported the same snapshot zip filename. Do not assign one zip to both runs. Add distinct preserved zip paths where available.
- `artifact does not contain paired *_pred and *_gt volumes`: rerun only evaluation and export for that experiment.
